# Dictionaries & Sets

Notes and runnable examples on `dict` and `set` — Python's hash-table types — and how CPython makes average-case **O(1)** lookup, insertion, and deletion happen under the hood.

**Contents**

1. **What they are** — hash tables, keys vs key→value
2. **CPython internals** — the open-addressing hash table & resizing
3. **The compact dict** — insertion order *and* less memory (3.6+)
4. **Hashing** — the `__hash__` / `__eq__` contract, and hash randomization
5. **`set` & `frozenset`** — a dict without values
6. **When to use what**
7. **Complexity cheat-sheet**

## 1. What they are

A `dict` maps **keys to values**; a `set` is the same machinery storing **keys only** (membership and de-duplication, no values). Both are backed by a **hash table**, which buys average-case **O(1)** lookup, insert, and delete: you hash the key to jump straight to its slot instead of scanning.

The price you pay for that speed: keys/elements must be **hashable** (roughly, immutable), there's no cheap ordering *by value*, and the table always carries spare capacity to stay fast.

## 2. CPython internals — the open-addressing hash table

CPython resolves collisions with **open addressing**, not chaining: every entry lives in one flat array, and when two keys hash to the same slot the table **probes** a sequence of other slots until it finds the key (or an empty slot). The probe sequence is deliberately scrambled with a "perturb" value so clustered hashes still spread out:

```
perturb = hash(key)
j = hash(key) % size
# on each collision:
j = (5*j + 1 + perturb) % size
perturb >>= 5
```

The table keeps itself **sparse for speed**: once it's about **2/3 full** it **resizes** to a larger power-of-two table and re-inserts everything, landing back at roughly 1/3 full. You can watch those resizes as jumps in `getsizeof` — the same idea as a list's growth, but triggered by *load factor* rather than length:

In [1]:
import sys

d = {}
prev = -1
for i in range(25):
    size = sys.getsizeof(d)
    if size != prev:
        print(f"len={len(d):2d}  getsizeof={size:5d}")
        prev = size
    d[i] = i


len= 0  getsizeof=   64
len= 1  getsizeof=  224
len= 6  getsizeof=  352
len=11  getsizeof=  632
len=22  getsizeof= 1168


The size holds steady while there's slack, then jumps at each resize (here around len 6, 11, 22 — each new table roughly doubles the slot count). Note an **empty `dict` is just the 64-byte object**: no hash table is allocated until the first insertion.

## 3. The compact dict — insertion order *and* less memory (3.6+)

Since Python 3.6 a `dict` keeps **insertion order** (promoted to a language guarantee in 3.7). That fell out of Raymond Hettinger's *compact dict* design, which splits the table in two:

- a small **indices array** — the actual hash table, holding *positions* (1, 2, 4, or 8 bytes each), and
- a dense **entries array** storing `(hash, key, value)` in **insertion order**, packed with no gaps.

A lookup hashes into the indices array to find a position, then reads the entry; iteration just walks the dense entries array top to bottom — which is why order is preserved for free. Storing small indices in the sparse table instead of full 24-byte entry slots also makes dicts **~20–25% smaller** than the pre-3.6 design. (A related optimization, **key-sharing dicts** / PEP 412, lets all instances of a class share one copy of their `__dict__` keys.)

Order is by **insertion**, not by hash or by sort:

In [2]:
d = {}
for word in ["banana", "apple", "cherry", "date"]:
    d[word] = len(word)

print("keys  :", list(d))            # insertion order - not sorted, not hash order
print("values:", list(d.values()))
d["banana"] = 99                     # updating a key keeps its original position
print("after update:", list(d.items()))


keys  : ['banana', 'apple', 'cherry', 'date']
values: [6, 5, 6, 4]
after update: [('banana', 99), ('apple', 5), ('cherry', 6), ('date', 4)]


## 4. Hashing & the `__hash__` / `__eq__` contract

A key works in a dict/set only if it's **hashable**, and two rules must hold:

1. **Equal objects must have equal hashes:** `a == b` implies `hash(a) == hash(b)`. (The reverse need not hold — distinct objects are allowed to collide.)
2. A key's hash must **not change** over its lifetime — which is exactly why mutable built-ins (`list`, `dict`, `set`) are unhashable.

Defining `__eq__` on a class makes Python **drop the default `__hash__`**, so instances become unhashable until you provide a consistent `__hash__` as well:

In [3]:
# __eq__ WITHOUT __hash__  ->  instances are unhashable, can't be keys
class PointBad:
    def __init__(self, x, y): self.x, self.y = x, y
    def __eq__(self, other):  return (self.x, self.y) == (other.x, other.y)

try:
    {PointBad(1, 2): "nope"}
except TypeError as e:
    print("unhashable:", e)

# Provide BOTH, consistently: equal objects -> equal hashes
class Point:
    def __init__(self, x, y): self.x, self.y = x, y
    def __eq__(self, other):  return (self.x, self.y) == (other.x, other.y)
    def __hash__(self):       return hash((self.x, self.y))

d = {Point(1, 2): "here"}
print("lookup by an equal-but-distinct object:", Point(1, 2) in d)   # True


unhashable: cannot use 'PointBad' as a dict key (unhashable type: 'PointBad')
lookup by an equal-but-distinct object: True


### Hash randomization

For `str` and `bytes`, `hash()` is **randomized per process** — CPython uses SipHash seeded with a random value at startup (since 3.4), as a defense against deliberate hash-collision DoS attacks. So the same string hashes to different values in different runs; never persist hash values or depend on a specific one. (`hash(int)` is *not* randomized — it's essentially the integer itself.)

In [4]:
import subprocess, sys

for _ in range(3):
    out = subprocess.run([sys.executable, "-c", "print(hash('hello'))"],
                         capture_output=True, text=True)
    print("hash('hello') in a fresh process:", out.stdout.strip())

print("hash(int) is stable across calls:  ", hash(42), hash(42))


hash('hello') in a fresh process: 2231110866351354399
hash('hello') in a fresh process: -819776594138464243
hash('hello') in a fresh process: -8335290147758125844
hash(int) is stable across calls:   42 42


## 5. `set` & `frozenset`

A `set` is a `dict` without values — the same open-addressing hash table — so membership, `add`, and `discard` are average **O(1)**, dramatically faster than scanning a `list`. `frozenset` is the **immutable** version: because it can't change, it's **hashable**, so it can be a dict key or an element of another set. Sets also give you fast algebra: union `|`, intersection `&`, difference `-`, symmetric difference `^`.

First, the payoff — `in` on a set vs a list of the same data:

In [5]:
import timeit

N = 100_000
as_list = list(range(N))
as_set  = set(as_list)
target  = N - 1                       # worst case for the list: scan all the way to the end

t_list = timeit.timeit(lambda: target in as_list, number=2000)
t_set  = timeit.timeit(lambda: target in as_set,  number=2000)
print(f"list  membership: {t_list*1000:8.2f} ms / 2000 lookups")
print(f"set   membership: {t_set*1000:8.4f} ms / 2000 lookups")
print(f"set is ~{t_list/t_set:.0f}x faster  (O(1) hash vs O(n) scan)")


list  membership:   676.08 ms / 2000 lookups
set   membership:   0.0477 ms / 2000 lookups
set is ~14164x faster  (O(1) hash vs O(n) scan)


In [6]:
a = {1, 2, 3, 4}
b = {3, 4, 5, 6}
print("a | b  (union)       :", a | b)
print("a & b  (intersection):", a & b)
print("a - b  (difference)  :", a - b)
print("a ^ b  (symmetric)   :", a ^ b)

# frozenset is immutable -> hashable -> can itself be a key or a set member
edge = frozenset({1, 2})
print("frozenset as a dict key:", {edge: "connects 1<->2"})


a | b  (union)       : {1, 2, 3, 4, 5, 6}
a & b  (intersection): {3, 4}
a - b  (difference)  : {1, 2}
a ^ b  (symmetric)   : {1, 2, 5, 6}
frozenset as a dict key: {frozenset({1, 2}): 'connects 1<->2'}


## 6. When to use what

| You need... | Reach for | Why |
|---|---|---|
| A key → value mapping | `dict` | average O(1) get/set/del, insertion-ordered |
| Membership / de-duplication (mutable) | `set` | O(1) `in`, set algebra; no values |
| A set or key that's itself hashable | `frozenset` | immutable ⇒ usable as a dict key / set member |
| To count occurrences | `collections.Counter` | `dict` subclass with tallying built in |
| A default value for missing keys | `collections.defaultdict` | auto-creates the default on first access |
| An ordered map | `dict` | order is guaranteed since 3.7; reach for `OrderedDict` only for its extras (e.g. `move_to_end`) |

## 7. Complexity cheat-sheet

| Operation | `dict` | `set` | `frozenset` | `list` (for contrast) |
|---|:---:|:---:|:---:|:---:|
| Lookup / membership | O(1) avg | O(1) avg | O(1) avg | O(n) |
| Insert / add | O(1) avg† | O(1) avg† | — (immutable) | O(1) append / O(n) insert |
| Delete / discard | O(1) avg | O(1) avg | — | O(n) |
| Iterate all | O(n) | O(n) | O(n) | O(n) |
| Lookup, worst case | O(n)‡ | O(n)‡ | O(n)‡ | O(n) |
| Memory per element | high | high | high | medium |

<sub>† amortized — an occasional O(n) resize when the table passes ~2/3 full. &middot; ‡ only under pathological hash collisions; with ordinary keys it's effectively O(1).</sub>